In [ ]:
!pip install timm soundfile -q

import ast, os, random, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import timm
import torch
import torch.nn as nn
import torchaudio
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset

print(f"torch={torch.__version__}, cuda={torch.cuda.is_available()}, timm={timm.__version__}")


In [ ]:
COMP_DIR = Path('/kaggle/input/birdclef-2026')
WORK     = Path('/kaggle/working')
TRAIN_CSV    = COMP_DIR / 'train.csv'
TAXONOMY_CSV = COMP_DIR / 'taxonomy.csv'
SS_LABELS_CSV = COMP_DIR / 'train_soundscapes_labels.csv'
AUDIO_DIR = COMP_DIR / 'train_audio'
SS_DIR    = COMP_DIR / 'train_soundscapes'
LABELS_NPZ = WORK / 'labels_v2.npz'
CACHE_NPY  = WORK / 'cache_v2.npy'
CACHE_META = WORK / 'cache_v2_meta.npz'
SR=32000; DUR_SEC=5.0; DUR=int(DUR_SEC*SR); HOP_AUDIO=SR
N_FFT=1024; HOP_LENGTH=320; N_MELS=128; F_MIN=50.0; F_MAX=14000.0
TOP_DB=80.0; NORM_MEAN=-4.268; NORM_STD=4.569; K_FOCAL=5
SECONDARY_WEIGHT=0.3
N_FOLDS=5; N_EPOCHS=30; BATCH_SIZE=32; LR=5e-4; EFFNET_DIM=1280
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")


In [ ]:
def build_labels():
    train = pd.read_csv(TRAIN_CSV)
    tax   = pd.read_csv(TAXONOMY_CSV)
    ts    = pd.read_csv(SS_LABELS_CSV)
    LABELS = sorted(tax['primary_label'].dropna().astype(str).unique())
    l2i = {l: i for i, l in enumerate(LABELS)}
    C   = len(LABELS)
    print(f"Classes: {C}")
    focal_paths, focal_labels, focal_strat = [], [], []
    for _, row in train.iterrows():
        p   = str(AUDIO_DIR / row['filename'])
        vec = np.zeros(C, dtype=np.float32)
        prim = str(row['primary_label'])
        if prim in l2i: vec[l2i[prim]] = 1.0
        try:
            sec = ast.literal_eval(row.get('secondary_labels','[]') or '[]')
        except: sec = []
        for s in sec:
            if s in l2i and vec[l2i[s]] < SECONDARY_WEIGHT:
                vec[l2i[s]] = SECONDARY_WEIGHT
        focal_paths.append(p); focal_labels.append(vec); focal_strat.append(prim)
    print(f"focal: {len(focal_paths)}")
    def pt(s):
        h,m,sec = s.split(':'); return int(h)*3600+int(m)*60+int(sec)
    ss_paths,ss_labels,ss_starts,ss_durs,ss_strat = [],[],[],[],[]
    for _, row in ts.iterrows():
        p = SS_DIR / row['filename']
        if not p.exists(): continue
        st=pt(str(row['start'])); en=pt(str(row['end']))
        vec=np.zeros(C,dtype=np.float32)
        valid=[s for s in str(row['primary_label']).split(';') if s in l2i]
        if not valid: continue
        for s in valid: vec[l2i[s]]=1.0
        ss_paths.append(str(p)); ss_labels.append(vec)
        ss_starts.append(float(st)); ss_durs.append(float(en-st)); ss_strat.append(valid[0])
    print(f"soundscape: {len(ss_paths)}")
    all_paths  = np.array(focal_paths+ss_paths, dtype=object)
    all_labels = np.stack(focal_labels+ss_labels)
    all_source = np.array(['focal']*len(focal_paths)+['soundscape']*len(ss_paths), dtype=object)
    all_starts = np.array([-1.]*len(focal_paths)+ss_starts, dtype=np.float32)
    all_durs   = np.array([-1.]*len(focal_paths)+ss_durs,   dtype=np.float32)
    all_strat  = np.array(focal_strat+ss_strat, dtype=object)
    np.savez_compressed(LABELS_NPZ, paths=all_paths, labels=all_labels, source=all_source,
        seg_start=all_starts, seg_dur=all_durs, primary_strat=all_strat,
        classes=np.array(LABELS, dtype=object))
    print(f"Saved {LABELS_NPZ} ({LABELS_NPZ.stat().st_size/1e6:.1f} MB)")
    return LABELS

LABELS = build_labels()


In [ ]:
def make_mel(): return torchaudio.transforms.MelSpectrogram(
    sample_rate=SR,n_fft=N_FFT,hop_length=HOP_LENGTH,n_mels=N_MELS,f_min=F_MIN,f_max=F_MAX)

def wav_to_spec(mel_mod, wav):
    if wav.dim()==1: wav=wav.unsqueeze(0)
    db=10.0*torch.log10(mel_mod(wav).clamp(min=1e-10))
    db=torch.maximum(db, db.flatten(-2).max(-1).values[...,None,None]-TOP_DB)
    return (db-NORM_MEAN)/NORM_STD

def topk_rms(wav, k=K_FOCAL):
    L=len(wav)
    if L<DUR: return [0]
    starts=np.arange(0,L-DUR+1,HOP_AUDIO)
    rms=np.array([float(np.sqrt(np.mean(wav[s:s+DUR]**2)+1e-12)) for s in starts])
    chosen=[]
    for idx in np.argsort(-rms):
        s=int(starts[idx])
        if all(abs(s-c)>=DUR for c in chosen): chosen.append(s)
        if len(chosen)>=k: break
    return chosen

def load_audio(p):
    w,sr=sf.read(p,dtype='float32')
    return w.mean(1) if w.ndim==2 else w

def build_cache():
    npz=np.load(LABELS_NPZ,allow_pickle=True)
    paths=npz['paths']; source=npz['source']; seg_start=npz['seg_start']
    n=len(paths)
    n_focal=int((source=='focal').sum()); n_ss=n-n_focal
    n_max=n_focal*K_FOCAL+n_ss
    T=DUR//HOP_LENGTH+1
    print(f"samples={n}(focal={n_focal},ss={n_ss}), max_windows={n_max}, ~{n_max*1*N_MELS*T*2/1e9:.1f}GB")
    tmp=WORK/'cache_tmp.npy'
    out=np.lib.format.open_memmap(tmp,mode='w+',dtype=np.float16,shape=(n_max,1,N_MELS,T))
    si_arr=np.empty(n_max,dtype=np.int32); ki_arr=np.empty(n_max,dtype=np.int8)
    rms_a=np.empty(n_max,dtype=np.float32); src_a=np.empty(n_max,dtype=object)
    mel=make_mel(); w=0; t0=time.time()
    for i in range(n):
        p=str(paths[i]); src=str(source[i])
        try:
            wav=load_audio(p)
            if src=='focal':
                if len(wav)<DUR: wav=np.pad(wav,(0,DUR-len(wav)))
                for ki,s in enumerate(topk_rms(wav)):
                    seg=wav[s:s+DUR]; sp=wav_to_spec(mel,torch.from_numpy(seg))
                    out[w]=sp.numpy().astype(np.float16); si_arr[w]=i; ki_arr[w]=ki
                    rms_a[w]=float(np.sqrt(np.mean(seg**2)+1e-12)); src_a[w]='focal'; w+=1
            else:
                s=int(float(seg_start[i])*SR)
                if len(wav)<s+DUR: wav=np.pad(wav,(0,s+DUR-len(wav)))
                seg=wav[s:s+DUR]; sp=wav_to_spec(mel,torch.from_numpy(seg))
                out[w]=sp.numpy().astype(np.float16); si_arr[w]=i; ki_arr[w]=0
                rms_a[w]=float(np.sqrt(np.mean(seg**2)+1e-12)); src_a[w]='soundscape'; w+=1
        except Exception as e: print(f"ERR {i}: {e}")
        if (i+1)%500==0:
            el=time.time()-t0; print(f"  {i+1}/{n} {el/60:.1f}m ETA={(el/(i+1)*(n-i-1))/60:.1f}m w={w}",flush=True)
    out.flush(); aw=w; print(f"windows={aw}")
    fin=np.lib.format.open_memmap(CACHE_NPY,mode='w+',dtype=np.float16,shape=(aw,1,N_MELS,T))
    fin[:]=out[:aw]; fin.flush(); del out,fin; tmp.unlink()
    np.savez_compressed(CACHE_META,sample_idx=si_arr[:aw],k_idx=ki_arr[:aw],
                        rms=rms_a[:aw],source=src_a[:aw].astype(object))
    print(f"Done: {CACHE_NPY} ({CACHE_NPY.stat().st_size/1e9:.2f}GB) in {(time.time()-t0)/60:.1f}min")

build_cache()


In [ ]:
class BirdCacheDataset(Dataset):
    def __init__(self, sample_indices, labels, window_lists, cache, is_train=True):
        self.sample_indices=sample_indices; self.labels=labels
        self.window_lists=window_lists; self.cache=cache; self.is_train=is_train
    def __len__(self): return len(self.sample_indices)
    def __getitem__(self, idx):
        wl=self.window_lists[idx]
        w=random.choice(wl) if (self.is_train and len(wl)>1) else wl[0]
        return torch.from_numpy(self.cache[w].astype(np.float32)),                torch.from_numpy(np.asarray(self.labels[idx],dtype=np.float32))

class SpecAugment(nn.Module):
    def __init__(self):
        super().__init__()
        self.fm=torchaudio.transforms.FrequencyMasking(30)
        self.tm=torchaudio.transforms.TimeMasking(40)
    def forward(self,x):
        for _ in range(2): x=self.fm(x)
        for _ in range(2): x=self.tm(x)
        return x

class Mixup(nn.Module):
    def __init__(self,alpha=1.0,theta=0.8): super().__init__(); self.alpha=alpha; self.theta=theta
    def forward(self,x,y):
        if not self.training or random.random()>self.theta: return x,y
        lam=float(np.random.beta(self.alpha,self.alpha)); idx=torch.randperm(x.size(0)).to(x.device)
        return lam*x+(1-lam)*x[idx], lam*y+(1-lam)*y[idx]

class EffNetMixup(nn.Module):
    def __init__(self,n_classes):
        super().__init__()
        self.spec_aug=SpecAugment(); self.mixup=Mixup()
        self.stem_conv=nn.Conv2d(1,3,3,1,1,bias=False)
        self.backbone=timm.create_model('tf_efficientnetv2_b0',pretrained=True,in_chans=3,num_classes=0)
        self.head=nn.Linear(EFFNET_DIM,n_classes)
    def forward(self,x,targets=None):
        if self.training:
            x=self.spec_aug(x)
            if targets is not None: x,targets=self.mixup(x,targets)
        return self.head(self.backbone(self.stem_conv(x))), targets

def set_seed(s=42): random.seed(s); np.random.seed(s); torch.manual_seed(s)
print("Classes ready")


In [ ]:
def train_fold(fold, ld, cache, meta):
    set_seed(42+fold)
    n=len(ld['paths'])
    wl=[[] for _ in range(n)]
    for w,si in enumerate(meta['sample_idx']): wl[int(si)].append(w)
    labels=ld['labels'].astype(np.float32)
    source=np.array([str(s) for s in ld['source']])
    pstrat=np.array([str(s) for s in ld['primary_strat']])
    fi=np.where(source=='focal')[0]; si=np.where(source!='focal')[0]
    skf=StratifiedKFold(N_FOLDS,shuffle=True,random_state=42)
    ftr,fval=list(skf.split(fi,pstrat[fi]))[fold]
    tr_idx=np.concatenate([fi[ftr],si]); va_idx=fi[fval]
    tr_ds=BirdCacheDataset(tr_idx,labels[tr_idx],[wl[i] for i in tr_idx],cache,True)
    va_ds=BirdCacheDataset(va_idx,labels[va_idx],[wl[i] for i in va_idx],cache,False)
    tr_ld=DataLoader(tr_ds,BATCH_SIZE,shuffle=True,num_workers=4,pin_memory=True,drop_last=True)
    va_ld=DataLoader(va_ds,BATCH_SIZE,shuffle=False,num_workers=4)
    n_cls=labels.shape[1]
    print(f"\n{'='*50}\nFold {fold+1}: Train={len(tr_ds)}(f={len(ftr)},ss={len(si)}),Val={len(va_ds)}")
    model=EffNetMixup(n_cls).to(DEVICE)
    crit=nn.BCEWithLogitsLoss()
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=N_EPOCHS,eta_min=1e-6)
    cp=WORK/f'ckpt_f{fold}.pth'; ep0=0; best=0.0
    if cp.exists():
        ck=torch.load(cp,map_location=DEVICE,weights_only=False)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['optimizer'])
        sch.load_state_dict(ck['scheduler']); ep0=ck['epoch']+1; best=ck['best']
    for ep in range(ep0,N_EPOCHS):
        t0=time.time(); model.train(); tl=0
        for sp,tgt in tr_ld:
            sp=sp.to(DEVICE); tgt=tgt.to(DEVICE)
            lg,tm=model(sp,tgt); loss=crit(lg,tm)
            opt.zero_grad(); loss.backward(); opt.step(); tl+=loss.item()
        sch.step()
        model.eval(); pa,ta=[],[]
        with torch.no_grad():
            for sp,tgt in va_ld:
                lg,_=model(sp.to(DEVICE)); pa.append(torch.sigmoid(lg).cpu().numpy()); ta.append(tgt.numpy())
        p=np.vstack(pa); t=np.vstack(ta)
        aucs=[roc_auc_score(t[:,j],p[:,j]) for j in range(n_cls) if t[:,j].sum()>0]
        auc=float(np.mean(aucs)) if aucs else 0.
        nb2=max(1,len(tr_ld))
        print(f"  Ep{ep+1}/{N_EPOCHS} loss={tl/nb2:.4f} auc={auc:.4f} ({time.time()-t0:.0f}s)",flush=True)
        if auc>best:
            best=auc; torch.save(model.state_dict(),WORK/f'best_fold{fold}.pth')
            print(f"    -> best {best:.4f}",flush=True)
        torch.save({'epoch':ep,'model':model.state_dict(),'optimizer':opt.state_dict(),
                    'scheduler':sch.state_dict(),'best':best},cp)
    if cp.exists(): cp.unlink()
    print(f"Fold {fold+1} best={best:.4f}"); return best

set_seed(42)
print(f"Loading cache: {CACHE_NPY}")
cache=np.load(str(CACHE_NPY),mmap_mode='r')
meta=np.load(str(CACHE_META),allow_pickle=True)
ld=dict(np.load(str(LABELS_NPZ),allow_pickle=True))
print(f"cache={cache.shape} labels={ld['labels'].shape}")
aucs=[]; tall=time.time()
for fold in range(N_FOLDS):
    bp=WORK/f'best_fold{fold}.pth'; ck=WORK/f'ckpt_f{fold}.pth'
    if bp.exists() and not ck.exists(): print(f"Fold {fold+1} done,skip"); aucs.append(-1.); continue
    aucs.append(train_fold(fold,ld,cache,meta))
done=[a for a in aucs if a>0]
if done:
    for i,a in enumerate(aucs):
        if a>0: print(f"  Fold {i+1}: {a:.4f}")
    print(f"  Mean: {np.mean(done):.4f}")
print(f"Total: {(time.time()-tall)/60:.1f}min")


In [ ]:
print("Output weights:")
for f in sorted(WORK.glob('best_fold*.pth')):
    print(f"  {f.name}  {f.stat().st_size/1e6:.1f} MB")
